In [2]:
import polars as pl
import pandas as pd
import glob
import json

# Simple clean, only merge all to 1 csv
## 1. Books

In [3]:
languages_df = pl.read_json('exports/language.json')
languages_df

id,language,code2
i64,str,str
1,"""English""","""en"""
2,"""Afar""","""aa"""
3,"""Abkhazian""","""ab"""
4,"""Afrikaans""","""af"""
5,"""Akan""","""ak"""
…,…,…
181,"""Yiddish""","""yi"""
182,"""Yoruba""","""yo"""
183,"""Zhuang; Chuang""","""za"""


In [4]:
file_paths = glob.glob("exports/good/hardcover_book*.json")

alldatas = []
for filepath in file_paths:
    with open(filepath, encoding='utf-8') as file:
        print(filepath)
        data = json.load(file)
        alldatas.extend(data)

exports/good\hardcover_book_1-50000.json
exports/good\hardcover_book_100001-150000.json
exports/good\hardcover_book_1100001-1150000.json
exports/good\hardcover_book_1150001-1200000.json
exports/good\hardcover_book_150001-200000.json
exports/good\hardcover_book_200001-250000.json
exports/good\hardcover_book_250001-300000.json
exports/good\hardcover_book_300001-350000.json
exports/good\hardcover_book_350001-400000.json
exports/good\hardcover_book_400001-450000.json
exports/good\hardcover_book_450001-500000.json
exports/good\hardcover_book_500001-550000.json
exports/good\hardcover_book_50001-100000.json
exports/good\hardcover_book_550001-600000.json
exports/good\hardcover_book_600001-650000.json
exports/good\hardcover_book_650001-700000.json
exports/good\hardcover_book_700001-750000.json


In [5]:
alldatas_fixed = []

for idx, data in enumerate(alldatas):
    if data["dto_combined"] == {} or data["dto_combined"].get("isbn_13", None) == None:
        # print(idx, data["id"])
        continue

    test_genres = []  # Default to Empty array
    cached_tags = [tag["tag"] for tag in data.get("cached_tags", {}).get("Genre", [])]
    dto_genres = data["dto_combined"].get("genres", [])
    if cached_tags or dto_genres:
        test_genres = list(dict.fromkeys(cached_tags + dto_genres))

    language_id = data["dto_combined"]["language_id"]
    
    result = {
        "id":           data["dto_combined"]["isbn_13"],
        "ISBN_10":      data["dto_combined"].get("isbn_10"),
        "title":        data["dto_combined"].get("title"),
        "description":  data["dto_combined"].get("description"),
        "image_url":    data["cached_image"].get("url"),
        "author":       data["cached_contributors"][0]["author"]["name"] if len(data["cached_contributors"]) > 0 else "",
        "language":     languages_df.row(1-1)[1] if language_id != None else "",
        "publish_date": data["dto_combined"].get("release_date"),
        "num_pages":    data["dto_combined"].get("page_count"),
        "crawl_source": "hardcover",
        "series": [series for series in data["dto_combined"]["series"]],
        "genre": test_genres,
    
        "editions":     data.get("editions", []),
        "hardcover_book_id": data.get("id")
    }
    alldatas_fixed.append(result)

    # except Exception as e:
    #     print(data["dto_combined"])

In [6]:
len(alldatas)

570000

In [7]:
len(alldatas_fixed)


223378

In [8]:
df_books = pd.DataFrame.from_records(alldatas_fixed)
df_books["num_pages"] = df_books["num_pages"].fillna(0).astype(int)
df_books

,id,ISBN_10,title,description,image_url,author,language,publish_date,num_pages,crawl_source,series,genre,editions,hardcover_book_id
0,9780713469875,0713469870,Pattern Drafting for Dressmaking,None,None,Pamela Stringer,English,1993-07-01,184,hardcover,[],[],[],1189157
1,9781776563531,1776563530,Head Girl,'The first time I read Freya’s work I thought ...,https://assets.hardcover.app/external_data/600...,Freya Daly Sadgrove,English,2020-01-01,107,hardcover,[],"[Young Adult, Poetry]",[],868944
2,9780811218832,081121883X,Everything and Nothing,A pocket-sized Pearls edition of some of Borge...,https://assets.hardcover.app/edition/30631767/...,Jorge Luis Borges,English,1999-04-01,129,hardcover,[],"[Classics, Literary Collections]",[],651199
3,9780450054594,0450054594,Cat's Revenge: More Than 101 Uses for Dead People,None,https://assets.hardcover.app/edition/31520632/...,Philip Lief,English,1981-01-01,98,hardcover,[],"[Comics, American wit and humor, Pictorial, A...",[],1474502
4,9781607051978,1607051974,Block Party - The Modern Quilting Bee: The Jou...,None,None,Alissa Haight Carlton,,2011-05-01,0,hardcover,[],[],[],1189155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223373,9780834842083,0834842084,"Gurdjieff Reconsidered: The Life, the Teaching...",None,None,Roger Lipsey,,2019-02-05,0,hardcover,[],[],[],1303747
223374,9780061288777,None,Celebrate Children's Day/book day =,Children read aloud in various settings to cel...,https://assets.hardcover.app/edition/4284569/2...,Pat Mora,English,2008-01-01,46,hardcover,[],"[Fiction, Young Adult, Juvenile Fiction, Spani...",[],6230
223375,9781732819214,None,Age of Asango: Book II,None,https://assets.hardcover.app/external_data/611...,Matt Russell,English,2018-11-16,425,hardcover,"[{'series_id': 58660, 'details': '2', 'positio...",[Fantasy],[],1424537
223376,9781781089101,1781089108,The Angel of the Crows,This is not the story you think it is. These a...,https://assets.hardcover.app/edition/7406604/1...,Katherine Addison,English,2020-06-23,464,hardcover,[],"[Fantasy, Adventure, Science fiction, LGBTQ, S...",[],448672


In [9]:
genres_df = df_books[["id", "genre"]].explode("genre").dropna(subset=["genre"])
genres_df

,id,genre
1,9781776563531,Young Adult
1,9781776563531,Poetry
2,9780811218832,Classics
2,9780811218832,Literary Collections
3,9780450054594,Comics
...,...,...
223376,9781781089101,Adventure
223376,9781781089101,Science fiction
223376,9781781089101,LGBTQ
223376,9781781089101,Science Fiction


In [10]:
genres_df.to_csv('exports/hardcover_book_genre.csv', index=False)

In [11]:
df_books.drop(columns=["genre","series","editions"]).to_csv('exports/hardcover_books.csv', index=False)

In [92]:
test_df = pl.scan_csv('exports/test.csv', schema={
    "id": pl.String,
    "ISBN_10": pl.String,
    "title": pl.String,
    "description": pl.String,
    "image_url": pl.String,
    "author": pl.String,
    "language": pl.String,
    "publish_date": pl.String,
    "num_pages": pl.UInt64,
    "crawl_source": pl.String
})

In [94]:
print(test_df.describe())
test_df.collect()

shape: (9, 11)
┌────────────┬───────────────┬────────────┬──────────────────┬───┬──────────┬──────────────┬────────────┬──────────────┐
│ statistic  ┆ id            ┆ ISBN_10    ┆ title            ┆ … ┆ language ┆ publish_date ┆ num_pages  ┆ crawl_source │
│ ---        ┆ ---           ┆ ---        ┆ ---              ┆   ┆ ---      ┆ ---          ┆ ---        ┆ ---          │
│ str        ┆ str           ┆ str        ┆ str              ┆   ┆ str      ┆ str          ┆ f64        ┆ str          │
╞════════════╪═══════════════╪════════════╪══════════════════╪═══╪══════════╪══════════════╪════════════╪══════════════╡
│ count      ┆ 193441        ┆ 171849     ┆ 193431           ┆ … ┆ 156687   ┆ 175590       ┆ 193441.0   ┆ 193441       │
│ null_count ┆ 0             ┆ 21592      ┆ 10               ┆ … ┆ 36754    ┆ 17851        ┆ 0.0        ┆ 0            │
│ mean       ┆ null          ┆ null       ┆ null             ┆ … ┆ null     ┆ null         ┆ 207.706505 ┆ null         │
│ std        ┆ nu

id,ISBN_10,title,description,image_url,author,language,publish_date,num_pages,crawl_source
str,str,str,str,str,str,str,str,u64,str
"""9780713469875""","""0713469870""","""Pattern Drafting for Dressmaki…",null,null,"""Pamela Stringer""","""English""","""1993-07-01""",184,"""hardcover"""
"""9781776563531""","""1776563530""","""Head Girl""","""'The first time I read Freya’s…","""https://assets.hardcover.app/e…","""Freya Daly Sadgrove""","""English""","""2020-01-01""",107,"""hardcover"""
"""9780811218832""","""081121883X""","""Everything and Nothing""","""A pocket-sized Pearls edition …","""https://assets.hardcover.app/e…","""Jorge Luis Borges""","""English""","""1999-04-01""",129,"""hardcover"""
"""9780450054594""","""0450054594""","""Cat's Revenge: More Than 101 U…",null,"""https://assets.hardcover.app/e…","""Philip Lief""","""English""","""1981-01-01""",98,"""hardcover"""
"""9781607051978""","""1607051974""","""Block Party - The Modern Quilt…",null,null,"""Alissa Haight Carlton""",null,"""2011-05-01""",0,"""hardcover"""
…,…,…,…,…,…,…,…,…,…
"""9780521379656""","""0521379652""","""The Neglect of Experiment""",null,"""https://assets.hardcover.app/e…","""Allan Franklin""","""English""","""1986-01-01""",304,"""hardcover"""
"""9781910281970""","""1910281972""","""Cabin Pressure: Zurich""",null,"""https://assets.hardcover.app/e…","""John Finnemore""","""English""","""2015-01-15""",0,"""hardcover"""
"""9780307460455""","""0307460452""","""Change we can believe in""","""Outlines Obama's plan to effec…","""https://assets.hardcover.app/b…","""Barack Obama""",null,"""2008-01-01""",273,"""hardcover"""


## 2. Users

In [68]:
file_paths = glob.glob("exports/hardcover_user*.json")

dfs = [pl.read_json(file) for file in file_paths]

In [69]:
combined_df = pl.concat(dfs, how="vertical_relaxed") #.lazy()
print(combined_df.describe())
combined_df

shape: (9, 5)
┌────────────┬──────────────┬───────┬────────────────┬─────────────┐
│ statistic  ┆ id           ┆ name  ┆ username       ┆ books_count │
│ ---        ┆ ---          ┆ ---   ┆ ---            ┆ ---         │
│ str        ┆ f64          ┆ str   ┆ str            ┆ f64         │
╞════════════╪══════════════╪═══════╪════════════════╪═════════════╡
│ count      ┆ 21336.0      ┆ 20885 ┆ 20840          ┆ 21336.0     │
│ null_count ┆ 0.0          ┆ 451   ┆ 496            ┆ 0.0         │
│ mean       ┆ 11864.202147 ┆ null  ┆ null           ┆ 103.32246   │
│ std        ┆ 6753.334542  ┆ null  ┆ null           ┆ 280.911759  │
│ min        ┆ 1.0          ┆       ┆ -DreamCatcher- ┆ 0.0         │
│ 25%        ┆ 6022.0       ┆ null  ┆ null           ┆ 0.0         │
│ 50%        ┆ 11814.0      ┆ null  ┆ null           ┆ 5.0         │
│ 75%        ┆ 17745.0      ┆ null  ┆ null           ┆ 83.0        │
│ max        ┆ 23482.0      ┆ 🪽    ┆ zzzorrio       ┆ 7951.0      │
└────────────┴───────

id,name,username,books_count
i64,str,str,i64
18783,"""Nitin Rughoonauth""","""Nitin""",7951
22985,"""Court""","""Courtreader""",7474
19573,"""Martha""","""Martha""",6933
13481,"""Deb Nance""","""DebNance""",6773
13318,"""abi""","""booksgamesvinyl""",5666
…,…,…,…
16898,"""Yaiza""","""Yai002""",0
14087,"""Stacy ""","""Tacy_0301""",0
2609,"""Thomas Bartensud""","""bartensud""",0


In [71]:
combined_df.lazy().sink_csv('exports/hardcover_users.csv')

## 3. Ratings

In [27]:
# Step 1: Get all JSON file paths
file_paths = glob.glob("exports/hardcover_rating*.json")

dfs = [pl.read_json(file) for file in file_paths]

In [28]:
combined_df = pl.concat(dfs, how="vertical_relaxed") #.lazy()
print(combined_df.describe())
combined_df

shape: (9, 6)
┌────────────┬────────────┬───────────────┬──────────────┬────────────┬─────────────┐
│ statistic  ┆ rating     ┆ book_id       ┆ user_id      ┆ date_added ┆ edition_id  │
│ ---        ┆ ---        ┆ ---           ┆ ---          ┆ ---        ┆ ---         │
│ str        ┆ f64        ┆ f64           ┆ f64          ┆ str        ┆ f64         │
╞════════════╪════════════╪═══════════════╪══════════════╪════════════╪═════════════╡
│ count      ┆ 1.631332e6 ┆ 1.631332e6    ┆ 1.631332e6   ┆ 1631332    ┆ 1.628765e6  │
│ null_count ┆ 0.0        ┆ 0.0           ┆ 0.0          ┆ 0          ┆ 2567.0      │
│ mean       ┆ 3.854516   ┆ 543281.600496 ┆ 10790.778025 ┆ null       ┆ 2.4687e7    │
│ std        ┆ 0.96229    ┆ 378920.3298   ┆ 6541.648061  ┆ null       ┆ 9.4193e6    │
│ min        ┆ 0.5        ┆ 1.0           ┆ 1.0          ┆ 0007-07-17 ┆ 179.0       │
│ 25%        ┆ 3.0        ┆ 326923.0      ┆ 5608.0       ┆ null       ┆ 1.9635348e7 │
│ 50%        ┆ 4.0        ┆ 440155.0    

rating,book_id,user_id,date_added,edition_id
f64,i64,i64,str,i64
3.0,1111086,6359,"""0007-07-17""",31135094
5.0,265718,20299,"""0008-01-18""",3537983
4.0,378793,20299,"""0008-02-14""",8631567
5.0,427391,20299,"""0008-02-14""",30193579
4.0,429541,20299,"""0008-02-14""",19508505
…,…,…,…,…
3.0,501689,11761,"""2021-04-19""",30451470
3.0,437921,10357,"""2021-04-19""",31263374
5.0,1238232,10357,"""2021-04-19""",31263253


In [29]:
combined_df.filter(pl.col("date_added").str.starts_with("00"))

rating,book_id,user_id,date_added,edition_id
f64,i64,i64,str,i64
3.0,1111086,6359,"""0007-07-17""",31135094
5.0,265718,20299,"""0008-01-18""",3537983
4.0,378793,20299,"""0008-02-14""",8631567
5.0,427391,20299,"""0008-02-14""",30193579
4.0,429541,20299,"""0008-02-14""",19508505
…,…,…,…,…
4.0,9724,19947,"""0024-10-17""",31227548
4.0,1494302,20764,"""0024-10-19""",31545933
3.0,427414,20764,"""0024-10-20""",12626703


In [30]:
combined_df.lazy().sink_csv('exports/hardcover_ratings.csv')

- CLEAN: Replace with 'Unknown' (maybe not)